In [1]:
from pathlib import Path
import sys
import gemmi
import json

In [ ]:
project_root = Path.cwd().parents[0]
print(project_root)
sys.path.insert(0, str(project_root))
from src.config.settings import settings


#"ABACUF_stripped.cif"
#"ABAGAO_stripped.cif"
cif_file = Path(settings.ready_cif_dir) / "AGAMUR_stripped.cif"

doc = gemmi.cif.read_file(str(cif_file))

json_str = doc.as_json()
d = json.loads(json_str)


# =========================
# Helper Functions
# =========================

def parse_cif_value_and_uncertainty(val_str):
    """
    Parses raw CIF values such as:
        '106.79(5)'
        '106.79'
        '15'
        '?'
        ['106.79(5)', '107.12(4)']

    Returns:
        (value, uncertainty) for a single value, or
        (values, uncertainties) for a list.
    """

    if val_str is None or val_str == '?':
        return None, 0.0

    # Handle lists recursively
    if isinstance(val_str, list):
        values = []
        uncertainties = []

        for item in val_str:
            value, uncertainty = parse_cif_value_and_uncertainty(item)
            values.append(value)
            uncertainties.append(uncertainty)

        return values, uncertainties

    val_str = str(val_str).strip()

    # No uncertainty reported
    if '(' not in val_str:
        return float(val_str), 0.0

    # Contains uncertainty
    main_val_str, unc_str = val_str.replace(')', '').split('(')

    if '.' in main_val_str:
        decimal_places = len(main_val_str.split(".")[1])
    else:
        decimal_places = 0

    uncertainty = int(unc_str) / (10 ** decimal_places)

    return float(main_val_str), uncertainty


# =========================
# Extract CIF variables
# =========================

# Get the first MOF entry in dictionary 'd'
mof_code = next(iter(d))

# Cell Volume & Formula Units
cell_volume, cell_volume_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_volume'))
cell_formula_units_z = d[mof_code].get('_cell_formula_units_z')

# Chemical
chemical_formula = d[mof_code].get('_chemical_formula_moiety')
chemical_name_systematic = d[mof_code].get('_chemical_name_systematic')

# Symmetry
symmetry_cell_setting = d[mof_code].get('_symmetry_cell_setting')
symmetry_space_group_name_h_m = d[mof_code].get('_symmetry_space_group_name_h-m')

symmetry_equiv_pos_site_id = d[mof_code].get('_symmetry_equiv_pos_site_id', [])
symmetry_equiv_pos_as_xyz = d[mof_code].get('_symmetry_equiv_pos_as_xyz', [])

symmetry_operations = [
    {
        "id": site_id,
        "operation": operation
    }
    for site_id, operation in zip(
        symmetry_equiv_pos_site_id,
        symmetry_equiv_pos_as_xyz
    )
]

# Cell lengths
cell_length_a, cell_length_a_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_length_a'))
cell_length_b, cell_length_b_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_length_b'))
cell_length_c, cell_length_c_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_length_c'))

# Cell angles
cell_angle_alpha, cell_angle_alpha_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_angle_alpha'))
cell_angle_beta, cell_angle_beta_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_angle_beta'))
cell_angle_gamma, cell_angle_gamma_uncertainty = parse_cif_value_and_uncertainty(d[mof_code].get('_cell_angle_gamma'))

# Atoms
atom_site_label = d[mof_code]['_atom_site_label']
atom_site_type_symbol = d[mof_code]['_atom_site_type_symbol']
atom_site_fract_x, atom_site_fract_x_uncertainty = parse_cif_value_and_uncertainty(d[mof_code]['_atom_site_fract_x'])
atom_site_fract_y, atom_site_fract_y_uncertainty= parse_cif_value_and_uncertainty(d[mof_code]['_atom_site_fract_y'])
atom_site_fract_z, atom_site_fract_z_uncertainty = parse_cif_value_and_uncertainty(d[mof_code]['_atom_site_fract_z'])


atoms_zipped = zip(
    atom_site_label,
    atom_site_type_symbol,
    atom_site_fract_x,
    atom_site_fract_y,
    atom_site_fract_z,
    atom_site_fract_x_uncertainty,
    atom_site_fract_y_uncertainty,
    atom_site_fract_z_uncertainty
    )




atoms = [
    {
        "label": label,
        "element": element,
        'x_coordinate': x,
        'x_coordinate_uncertainty': x_unc,
        'y_coordinate': y,
        'y_coordinate_uncertainty': y_unc,
        'z_coordinate': z,      
        'z_coordinate_uncertainty': z_unc
    }
    for label, element, x, y, z, x_unc, y_unc, z_unc in atoms_zipped
]



cell = gemmi.UnitCell(
    float(cell_length_a),
    float(cell_length_b),
    float(cell_length_c),
    float(cell_angle_alpha),
    float(cell_angle_beta),
    float(cell_angle_gamma)
)
 
 
atoms = []
 
for label, element, x, y, z in zip(
    atom_site_label,
    atom_site_type_symbol,
    atom_site_fract_x,
    atom_site_fract_y,
    atom_site_fract_z
):

 
    # Fractional coordinates
    frac = gemmi.Fractional(
        float(x),
        float(y),
        float(z)
    )
 
    # Convert to Cartesian Angstrom coordinates
    cart = cell.orthogonalize(frac)
 
    atoms.append(
        {
            "label": label,
            "element": element,
 
            "fractional_coordinates": {
                "x": float(x),
                "y": float(y),
                "z": float(z)
            },
 
            "cartesian_coordinates": {
                "x": round(cart.x, 4),
                "y": round(cart.y, 4),
                "z": round(cart.z, 4),
                "unit": "angstrom"
            }
        }
    )

# Bonds
geom_bond_atom_site_label_1 = d[mof_code].get('_geom_bond_atom_site_label_1', [])
geom_bond_atom_site_label_2 = d[mof_code].get('_geom_bond_atom_site_label_2', [])
geom_bond_site_symmetry_1 = d[mof_code].get('_geom_bond_site_symmetry_1', [])
geom_bond_site_symmetry_2 = d[mof_code].get('_geom_bond_site_symmetry_2', [])

bonds = [
    {
        "atom_1": atom_1,
        "atom_2": atom_2,
        "symmetry": {
            "atom_1": symmetry_1,
            "atom_2": symmetry_2
        }
    }
    for atom_1, symmetry_1, atom_2, symmetry_2 in zip(
        geom_bond_atom_site_label_1,
        geom_bond_site_symmetry_1,
        geom_bond_atom_site_label_2,
        geom_bond_site_symmetry_2
    )
]

# =========================
# Construct Data Structure
# =========================

mof_data = {
    "mof_code": mof_code,
    "chemical": {
        "formula": chemical_formula,
        "name_systematic": chemical_name_systematic
    },
    "cell": {
        "length_a": cell_length_a,
        "length_a_uncertainty": cell_length_a_uncertainty,
        "length_b": cell_length_b,
        "length_b_uncertainty": cell_length_b_uncertainty,
        "length_c": cell_length_c,
        "length_c_uncertainty": cell_length_c_uncertainty,
        "angle_alpha": cell_angle_alpha,
        "angle_alpha_uncertainty": cell_angle_alpha_uncertainty,
        "angle_beta": cell_angle_beta,
        "angle_beta_uncertainty": cell_angle_beta_uncertainty,
        "angle_gamma": cell_angle_gamma,
        "angle_gamma_uncertainty": cell_angle_gamma_uncertainty,
        "volume": cell_volume,
        "formula_units_z": cell_formula_units_z
    },
    "symmetry": {
        "cell_setting": symmetry_cell_setting,
        "space_group_h_m": symmetry_space_group_name_h_m,
        "operations": symmetry_operations
    },
    "atoms": atoms,
    "bonds": bonds
}


# =========================
# Export to JSON File
# =========================

print(json.dumps(mof_data, indent=4))

/Users/jamesgodfrey/Documents/Code/knowledgegraphRAG
{
    "mof_code": "agamur",
    "chemical": {
        "formula": "(C14 H20 Mn1 N8 1+)n,n(C6 N4 1-)",
        "name_systematic": "catena-((\\m~2~-gem-Tetracyanoethylene radical-N,N')-(1,4,7,10-tetra-azacyclododecane)-manganese(ii) tetracyanoethylene radical anion)"
    },
    "cell": {
        "length_a": 10.8895,
        "length_a_uncertainty": 0.0019,
        "length_b": 11.7647,
        "length_b_uncertainty": 0.0017,
        "length_c": 16.981,
        "length_c_uncertainty": 0.004,
        "angle_alpha": 90.0,
        "angle_alpha_uncertainty": 0.0,
        "angle_beta": 90.0,
        "angle_beta_uncertainty": 0.0,
        "angle_gamma": 90.0,
        "angle_gamma_uncertainty": 0.0,
        "volume": 2175.46,
        "formula_units_z": 4
    },
    "symmetry": {
        "cell_setting": "orthorhombic",
        "space_group_h_m": "P 21 21 21",
        "operations": [
            {
                "id": 1,
                "operation